In [10]:
# =============================================================================
# CropQuery — Phase C Training
# Full backbone unfreeze with layerwise LR decay
# Starting point: phase2_best.pth
# Target: >0.91 Top-1
# Hardware: Kaggle T4 x2 (DataParallel)
# =============================================================================

import os
import json
import time
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
import kagglehub

# =============================================================================
# DATASET DOWNLOADS
# =============================================================================

tushar5harma_path    = kagglehub.dataset_download('tushar5harma/plant-village-dataset-updated')
shubham2703_path     = kagglehub.dataset_download('shubham2703/five-crop-diseases-dataset')
zunorain_path        = kagglehub.dataset_download('zunorain/pea-plant-dataset')
mexwell_path         = kagglehub.dataset_download('mexwell/crop-diseases-classification')
ayushbpanchal_path   = kagglehub.dataset_download('ayushbpanchal/cashew-image-dataset')
tahmidmir_path       = kagglehub.dataset_download('tahmidmir/pumpkin-leaf-diseases-dataset-from-bangladesh')
binnassor89_path     = kagglehub.dataset_download('binnassor89/melon-macronutrient-deficiency-dataset')
nirmalsankalana_path = kagglehub.dataset_download('nirmalsankalana/rocole-a-robusta-coffee-leaf-images-dataset')

print('Downloads complete.')

Downloads complete.


In [11]:

# =============================================================================
# CONFIG
# =============================================================================

PHASE_B_CHECKPOINT = "/kaggle/input/models/vitthaljauhari/resnet50-phase2-pth/pytorch/default/1/phase2_best.pth"
RUNS_DIR           = "/kaggle/working/runs"
os.makedirs(RUNS_DIR, exist_ok=True)

BASE        = "/kaggle/input/datasets"   # rglob root per dataset
IMAGE_EXTS  = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

BATCH_SIZE          = 128
NUM_WORKERS         = 8
SEED                = 42
N_EPOCHS            = 15
IMG_SIZE            = 224
EARLY_STOP_PATIENCE = 3

LR_MAP = {
    "conv1_bn1": 1e-5,
    "layer1":    1e-5,
    "layer2":    2e-5,
    "layer3":    3e-5,
    "layer4":    5e-5,
    "fc":        5e-5,
}

torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} | GPUs: {torch.cuda.device_count()}")

# =============================================================================
# LABEL MAPS  (parent folder name → canonical)
# Matches Phase B notebook exactly — rglob uses img_path.parent.name
# =============================================================================

dataset_label_maps = {

    # tushar5harma: keyed by (crop_dir, disease_dir) — 3-level: Crop/Split/Disease
    "tushar5harma": {
        ("Apple",        "Apple Scab")             : "Apple__Apple_Scab",
        ("Apple",        "Black Rot")              : "Apple__Black_Rot",
        ("Apple",        "Cedar Apple Rust")       : "Apple__Cedar_Apple_Rust",
        ("Apple",        "Healthy")                : "Apple__Healthy",
        ("Bell Pepper",  "Bacterial Spot")         : "Bell_Pepper__Bacterial_Spot",
        ("Bell Pepper",  "Healthy")                : "Bell_Pepper__Healthy",
        ("Cherry",       "Healthy")                : "Cherry__Healthy",
        ("Cherry",       "Powdery Mildew")         : "Cherry__Powdery_Mildew",
        ("Corn (Maize)", "Cercospora Leaf Spot")   : "Corn__Gray_Leaf_Spot",
        ("Corn (Maize)", "Common Rust")            : "Corn__Common_Rust",
        ("Corn (Maize)", "Healthy")                : "Corn__Healthy",
        ("Corn (Maize)", "Northern Leaf Blight")   : "Corn__Northern_Leaf_Blight",
        ("Grape",        "Black Rot")              : "Grape__Black_Rot",
        ("Grape",        "Esca (Black Measles)")   : "Grape__Black_Measles",
        ("Grape",        "Healthy")                : "Grape__Healthy",
        ("Grape",        "Leaf Blight")            : "Grape__Leaf_Blight",
        ("Peach",        "Bacterial Spot")         : "Peach__Bacterial_Spot",
        ("Peach",        "Healthy")                : "Peach__Healthy",
        ("Potato",       "Early Blight")           : "Potato__Early_Blight",
        ("Potato",       "Healthy")                : "Potato__Healthy",
        ("Potato",       "Late Blight")            : "Potato__Late_Blight",
        ("Strawberry",   "Healthy")                : "Strawberry__Healthy",
        ("Strawberry",   "Leaf Scorch")            : "Strawberry__Leaf_Scorch",
        ("Tomato",       "Bacterial Spot")         : "Tomato__Bacterial_Spot",
        ("Tomato",       "Early Blight")           : "Tomato__Early_Blight",
        ("Tomato",       "Healthy")                : "Tomato__Healthy",
        ("Tomato",       "Late Blight")            : "Tomato__Late_Blight",
        ("Tomato",       "Septoria Leaf Spot")     : "Tomato__Septoria_Leaf_Spot",
        ("Tomato",       "Yellow Leaf Curl Virus") : "Tomato__Yellow_Leaf_Curl_Virus",
    },

    # shubham2703: structure is Crop Diseases/Crop___Disease/{Crop}/{Class}/*.jpg
    # rglob parent.name gives the leaf class folder name e.g. "Corn___Common_Rust"
    "shubham2703": {
        "Corn___Common_Rust"          : "Corn__Common_Rust",
        "Corn___Gray_Leaf_Spot"       : "Corn__Gray_Leaf_Spot",
        "Corn___Healthy"              : "Corn__Healthy",
        "Corn___Northern_Leaf_Blight" : "Corn__Northern_Leaf_Blight",
        "Potato___Early_Blight"       : "Potato__Early_Blight",
        "Potato___Healthy"            : "Potato__Healthy",
        "Potato___Late_Blight"        : "Potato__Late_Blight",
        "Rice___Brown_Spot"           : "Rice__Brown_Spot",
        "Rice___Healthy"              : "Rice__Healthy",
        "Rice___Leaf_Blast"           : "Rice__Leaf_Blast",
        "Rice___Neck_Blast"           : "Rice__Neck_Blast",
        "Wheat___Brown_Rust"          : "Wheat__Brown_Rust",
        "Wheat___Healthy"             : "Wheat__Healthy",
        "Wheat___Yellow_Rust"         : "Wheat__Yellow_Rust",
        "Bacterial Blight"            : "Rice__Bacterial_Blight",
        "Red Rot"                     : "Sugarcane__Red_Rot",
        # "Healthy" omitted — ambiguous, no crop prefix
    },

    # nirmalsankalana: only coffee classes present on disk
    "nirmalsankalana": {
        "coffee___healthy"          : "Coffee__Healthy",
        "coffee___red_spider_mite"  : "Coffee__Red_Spider_Mite",
        "coffee___rust"             : "Coffee__Rust",
    },

    # binnassor89: actual folder names confirmed from disk
    "binnassor89": {
        "calcium"   : "Melon__Calcium_Deficiency",
        "health"    : "Melon__Healthy",
        "nitrogen"  : "Melon__Nitrogen_Deficiency",
        "potasium"  : "Melon__Potassium_Deficiency",   # sic — matches disk
    },

    # zunorain: actual folder names confirmed from disk
    "zunorain": {
        "DOWNY_MILDEW_LEAF"  : "Pea__Downy_Mildew",
        "FRESH_LEAF"         : "Pea__Healthy",
        "LEAFMINNER_LEAF"    : "Pea__Leaf_Miner",
        "POWDER_MILDEW_LEAF" : "Pea__Powdery_Mildew",
    },

    # ayushbpanchal: train_set + test_set both present; rglob picks both
    "ayushbpanchal": {
        "anthracnose" : "Cashew__Anthracnose",
        "gumosis"     : "Cashew__Gummosis",
        "healthy"     : "Cashew__Healthy",
        "leaf miner"  : "Cashew__Leaf_Miner",
        "red rust"    : "Cashew__Red_Rust",
    },

    # tahmidmir: Original Dataset/{Class}/*.jpg — rglob parent.name = class folder
    "tahmidmir": {
        "Bacterial Leaf Spot" : "Pumpkin__Bacterial_Leaf_Spot",
        "Downy Mildew"        : "Pumpkin__Downy_Mildew",
        "Healthy Leaf"        : "Pumpkin__Healthy",
        "Mosaic Disease"      : "Pumpkin__Mosaic_Disease",
        "Powdery_Mildew"      : "Pumpkin__Powdery_Mildew",
    },

    # mexwell: CSV-based — label resolved via label_num_to_disease_map.json
    "mexwell": {
        "Cassava Bacterial Blight (CBB)"      : "Cassava__Bacterial_Blight",
        "Cassava Brown Streak Disease (CBSD)" : "Cassava__Brown_Streak_Disease",
        "Cassava Green Mottle (CGM)"          : "Cassava__Green_Mottle",
        "Cassava Mosaic Disease (CMD)"        : "Cassava__Mosaic_Disease",
        "Healthy"                             : "Cassava__Healthy",
    },
}

# Build global class_to_idx  (must match Phase B exactly — same label maps)
all_canonical = sorted(set(
    v for mapping in dataset_label_maps.values()
    for v in mapping.values()
    if v is not None
))
class_to_idx = {cls: idx for idx, cls in enumerate(all_canonical)}
N_CLASSES    = len(class_to_idx)
print(f"Total canonical classes: {N_CLASSES}")

Device: cuda | GPUs: 2
Total canonical classes: 64


In [12]:

 # =============================================================================
# LOADERS  — rglob strategy matches Phase B notebook
# =============================================================================

def build_samples_folder(slug: str) -> list:
    """
    rglob over the dataset root; match on img_path.parent.name.
    Handles arbitrary nesting (shubham2703's 4-level, ayushbpanchal's train/test, etc.)
    """
    label_map = dataset_label_maps[slug]
    root      = Path(BASE) / slug
    samples, skipped = [], 0
    for img_path in root.rglob("*"):
        if img_path.suffix.lower() not in IMAGE_EXTS:
            continue
        canonical = label_map.get(img_path.parent.name)
        if canonical is None:
            skipped += 1
            continue
        samples.append((str(img_path), class_to_idx[canonical]))
    print(f"  [{slug:<20}] {len(samples):>6} samples  ({skipped} skipped)")
    return samples


def build_samples_tushar() -> list:
    """
    Structure: {Crop}/{Split}/{Disease}/*.jpg
    Key = (crop_dir.name, disease_dir.name)
    """
    label_map = dataset_label_maps["tushar5harma"]
    root      = Path(BASE) / "tushar5harma"
    samples, skipped = [], 0
    for img_path in root.rglob("*"):
        if img_path.suffix.lower() not in IMAGE_EXTS:
            continue
        # img_path.parents: [disease_dir, split_dir, crop_dir, root, ...]
        disease_dir = img_path.parent
        split_dir   = disease_dir.parent
        crop_dir    = split_dir.parent
        key = (crop_dir.name, disease_dir.name)
        canonical = label_map.get(key)
        if canonical is None:
            skipped += 1
            continue
        samples.append((str(img_path), class_to_idx[canonical]))
    print(f"  [{'tushar5harma':<20}] {len(samples):>6} samples  ({skipped} skipped)")
    return samples


def build_samples_mexwell() -> list:
    """
    CSV-based. Label resolved via label_num_to_disease_map.json (int key → disease name).
    """
    data_root = Path(BASE) / "mexwell" / "crop-diseases-classification" / "Data"
    img_dir   = data_root / "train_images"
    json_path = data_root / "label_num_to_disease_map.json"
    csv_path  = data_root / "train.csv"

    with open(json_path) as f:
        num_to_name = json.load(f)   # {"0": "Cassava Bacterial Blight (CBB)", ...}

    df        = pd.read_csv(csv_path)
    label_map = dataset_label_maps["mexwell"]
    samples, skipped = [], 0

    for _, row in df.iterrows():
        img_path     = img_dir / row["image_id"]
        if not img_path.exists():
            skipped += 1
            continue
        disease_name = num_to_name[str(row["label"])]
        canonical    = label_map.get(disease_name)
        if canonical is None:
            skipped += 1
            continue
        samples.append((str(img_path), class_to_idx[canonical]))

    print(f"  [{'mexwell':<20}] {len(samples):>6} samples  ({skipped} skipped)")
    return samples


def build_all_samples():
    print("Loading datasets...")
    folder_slugs = ["shubham2703", "nirmalsankalana", "binnassor89",
                    "zunorain", "ayushbpanchal", "tahmidmir"]
    all_samples = []
    for slug in folder_slugs:
        all_samples += build_samples_folder(slug)
    all_samples += build_samples_mexwell()
    all_samples += build_samples_tushar()
    print(f"\nTotal: {len(all_samples):,} samples | "
          f"{len(set(lbl for _, lbl in all_samples))} unique class indices")
    return all_samples

In [13]:
# =============================================================================
# TRANSFORMS
# =============================================================================

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

In [14]:
# =============================================================================
# DATASET
# =============================================================================

class CropDiseaseDataset(Dataset):
    """samples: list of (path_str, class_idx_int) — same format as Phase B."""
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

In [15]:
# =============================================================================
# DATALOADERS
# =============================================================================

def build_dataloaders(all_samples):
    # Stratify on label index
    labels     = [lbl for _, lbl in all_samples]
    train_s, val_s = train_test_split(
        all_samples, test_size=0.15, random_state=SEED, stratify=labels
    )

    train_ds = CropDiseaseDataset(train_s, transform=train_transform)
    val_ds   = CropDiseaseDataset(val_s,   transform=val_transform)

    # WeightedRandomSampler
    train_labels  = [lbl for _, lbl in train_s]
    class_counts  = np.bincount(train_labels, minlength=N_CLASSES)
    class_weights = 1.0 / (class_counts + 1e-6)
    sample_w      = [class_weights[l] for l in train_labels]
    sampler       = WeightedRandomSampler(sample_w, num_samples=len(sample_w), replacement=True)

    kw = dict(num_workers=NUM_WORKERS, prefetch_factor=4,
              persistent_workers=True, pin_memory=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, **kw)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,   **kw)

    print(f"Train: {len(train_ds):,}  |  Val: {len(val_ds):,}")
    return train_loader, val_loader

In [16]:
# =============================================================================
# MODEL
# =============================================================================

def load_phase_b_model(path):
    ckpt = torch.load(path, map_location="cpu")
    assert ckpt["n_classes"] == N_CLASSES, \
        f"Checkpoint n_classes={ckpt['n_classes']} but label maps give {N_CLASSES}. Mismatch."

    model = models.resnet50(weights=None)
    model.fc = nn.Sequential(nn.Dropout(0.4), nn.Linear(2048, N_CLASSES))

    state = ckpt["model_state_dict"]
    if any(k.startswith("module.") for k in state):
        state = {k.replace("module.", "", 1): v for k, v in state.items()}
    model.load_state_dict(state)
    print(f"Loaded Phase B | epoch={ckpt['epoch']} | best_acc={ckpt['best_acc']:.4f}")
    return model, ckpt


def build_optimizer(model):
    for p in model.parameters():
        p.requires_grad = True
    base = model.module if hasattr(model, "module") else model
    return optim.Adam([
        {"params": list(base.conv1.parameters()) + list(base.bn1.parameters()),
         "lr": LR_MAP["conv1_bn1"], "name": "conv1_bn1"},
        {"params": base.layer1.parameters(), "lr": LR_MAP["layer1"], "name": "layer1"},
        {"params": base.layer2.parameters(), "lr": LR_MAP["layer2"], "name": "layer2"},
        {"params": base.layer3.parameters(), "lr": LR_MAP["layer3"], "name": "layer3"},
        {"params": base.layer4.parameters(), "lr": LR_MAP["layer4"], "name": "layer4"},
        {"params": base.fc.parameters(),     "lr": LR_MAP["fc"],     "name": "fc"},
    ])


In [17]:
# =============================================================================
# TRAIN / EVAL
# =============================================================================

def accuracy_topk(output, target, topk=(1, 5)):
    with torch.no_grad():
        maxk = max(topk)
        bs   = target.size(0)
        _, pred = output.topk(maxk, dim=1, largest=True, sorted=True)
        correct = pred.t().eq(target.view(1, -1).expand_as(pred.t()))
        return [correct[:k].reshape(-1).float().sum().mul_(100.0 / bs).item() for k in topk]


def train_one_epoch(model, loader, criterion, optimizer, epoch):
    model.train()
    loss_sum = top1_sum = top5_sum = 0.0
    t0 = time.time()
    for i, (imgs, labels) in enumerate(loader):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        top1, top5 = accuracy_topk(out, labels)
        loss_sum += loss.item(); top1_sum += top1; top5_sum += top5
        if (i + 1) % 50 == 0:
            print(f"  [E{epoch} {i+1}/{len(loader)}] "
                  f"loss={loss_sum/(i+1):.4f}  top1={top1_sum/(i+1):.2f}%  "
                  f"t={time.time()-t0:.0f}s")
    n = len(loader)
    return loss_sum / n, top1_sum / n, top5_sum / n


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    t1 = t5 = total = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out = model(imgs)
        a1, a5 = accuracy_topk(out, labels)
        bs = labels.size(0)
        t1 += a1 * bs; t5 += a5 * bs; total += bs
    return t1 / total, t5 / total

# =============================================================================
# CHECKPOINT
# =============================================================================

def save_checkpoint(model, optimizer, scheduler, epoch, val_top1, tag):
    base = model.module if hasattr(model, "module") else model
    path = os.path.join(RUNS_DIR, f"phaseC_{tag}.pth")
    torch.save({
        "epoch":                epoch,
        "phase":                "C",
        "model_state_dict":     base.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_acc":             val_top1,
        "class_to_idx":         class_to_idx,
        "n_classes":            N_CLASSES,
    }, path)
    print(f"  → phaseC_{tag}.pth  (top1={val_top1:.4f})")

# =============================================================================
# MAIN
# =============================================================================

def main():
    # 1. Model
    model, ckpt = load_phase_b_model(PHASE_B_CHECKPOINT)
    if torch.cuda.device_count() > 1:
        print(f"DataParallel: {torch.cuda.device_count()} GPUs")
        model = nn.DataParallel(model)
    model = model.to(device)

    # 2. Data
    all_samples              = build_all_samples()
    train_loader, val_loader = build_dataloaders(all_samples)

    # 3. Optimizer  (Phase B had 3 param groups; Phase C has 6 — fresh state expected)
    optimizer = build_optimizer(model)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS, eta_min=1e-7)
    criterion = nn.CrossEntropyLoss()

    best_top1  = ckpt.get("best_acc", 0.0)
    no_improve = 0
    history    = []

    print(f"\n{'='*60}")
    print(f"Phase C — Full Backbone Unfreeze")
    print(f"Baseline top-1: {best_top1:.4f} | Target: >0.91 | Patience: {EARLY_STOP_PATIENCE}")
    print(f"{'='*60}")

    for epoch in range(N_EPOCHS):
        lr_str = " | ".join(f"{g['name']}={g['lr']:.1e}" for g in optimizer.param_groups)
        print(f"\n--- Epoch {epoch}/{N_EPOCHS-1}  [{lr_str}] ---")

        tr_loss, tr_t1, tr_t5 = train_one_epoch(model, train_loader, criterion, optimizer, epoch)
        val_t1, val_t5         = evaluate(model, val_loader)
        scheduler.step()

        history.append({"epoch": epoch, "loss": round(tr_loss, 4),
                         "tr_t1": round(tr_t1, 4), "val_t1": round(val_t1, 4),
                         "val_t5": round(val_t5, 4)})

        flag = ""
        if val_t1 > best_top1:
            best_top1  = val_t1
            no_improve = 0
            save_checkpoint(model, optimizer, scheduler, epoch, val_t1, "best")
            flag = "  ← best"
        else:
            no_improve += 1

        print(f"Epoch {epoch:02d} | loss={tr_loss:.4f} | "
              f"train={tr_t1:.2f}% | val_t1={val_t1:.2f}% | val_t5={val_t5:.2f}%{flag}")

        if tr_t1 - val_t1 > 8.0:
            print(f"  ⚠ gap={tr_t1 - val_t1:.1f}pp")
        if val_t1 >= 91.0:
            print("✅ Target reached.")
            break
        if no_improve >= EARLY_STOP_PATIENCE:
            print(f"⏹ Early stop ({EARLY_STOP_PATIENCE} epochs no improvement).")
            break

    save_checkpoint(model, optimizer, scheduler, epoch, val_t1, "final")

    print(f"\n{'='*60}\nPhase C Summary")
    print(f"{'Ep':>3} | {'Loss':>7} | {'Train%':>7} | {'Val T1':>7} | {'Val T5':>7}")
    for r in history:
        print(f"{r['epoch']:>3} | {r['loss']:>7.4f} | {r['tr_t1']:>7.2f} | "
              f"{r['val_t1']:>7.2f} | {r['val_t5']:>7.2f}")
    print(f"\nBest Val Top-1: {best_top1:.2f}%")


if __name__ == "__main__":
    main()

Loaded Phase B | epoch=20 | best_acc=0.8072
DataParallel: 2 GPUs
Loading datasets...
  [shubham2703         ]  13224 samples  (100 skipped)
  [nirmalsankalana     ]   1560 samples  (1600 skipped)
  [binnassor89         ]    200 samples  (0 skipped)
  [zunorain            ]   1432 samples  (0 skipped)
  [ayushbpanchal       ]  25811 samples  (0 skipped)
  [tahmidmir           ]   2000 samples  (0 skipped)
  [mexwell             ]  17938 samples  (3459 skipped)
  [tushar5harma        ]  64727 samples  (2391 skipped)

Total: 126,892 samples | 64 unique class indices
Train: 107,858  |  Val: 19,034

Phase C — Full Backbone Unfreeze
Baseline top-1: 0.8072 | Target: >0.91 | Patience: 3

--- Epoch 0/14  [conv1_bn1=1.0e-05 | layer1=1.0e-05 | layer2=2.0e-05 | layer3=3.0e-05 | layer4=5.0e-05 | fc=5.0e-05] ---


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


  [E0 50/843] loss=0.6467  top1=79.75%  t=92s
  [E0 100/843] loss=0.6259  top1=80.17%  t=183s
  [E0 150/843] loss=0.6149  top1=80.26%  t=253s
  [E0 200/843] loss=0.6117  top1=80.12%  t=324s
  [E0 250/843] loss=0.6035  top1=80.22%  t=408s
  [E0 300/843] loss=0.6030  top1=80.14%  t=482s
  [E0 350/843] loss=0.6024  top1=80.06%  t=552s
  [E0 400/843] loss=0.5996  top1=80.12%  t=623s
  [E0 450/843] loss=0.5983  top1=80.08%  t=703s
  [E0 500/843] loss=0.5965  top1=80.00%  t=777s
  [E0 550/843] loss=0.5930  top1=80.07%  t=848s
  [E0 600/843] loss=0.5914  top1=80.05%  t=927s
  [E0 650/843] loss=0.5899  top1=80.06%  t=1010s
  [E0 700/843] loss=0.5881  top1=80.07%  t=1077s
  [E0 750/843] loss=0.5858  top1=80.10%  t=1155s
  [E0 800/843] loss=0.5837  top1=80.14%  t=1235s
  → phaseC_best.pth  (top1=78.8852)
Epoch 00 | loss=0.5820 | train=80.13% | val_t1=78.89% | val_t5=98.58%  ← best

--- Epoch 1/14  [conv1_bn1=9.9e-06 | layer1=9.9e-06 | layer2=2.0e-05 | layer3=3.0e-05 | layer4=4.9e-05 | fc=4.9e-05